In [1]:
%config InlineBackend.figure_format = 'svg'

In [2]:
import os
import random
import time 
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from node2vec import Node2Vec
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from torch_geometric.data import Data
import spektral
from spektral.layers import GCNConv, GATConv
from spektral.layers import GraphSageConv
from spektral.data import Graph, Dataset, BatchLoader
from scipy.sparse import csr_matrix
from torch_geometric.nn import DeepGraphInfomax, VGAE
from torch_geometric.utils import from_networkx
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from collections import Counter
from sklearn.preprocessing import normalize
from joblib import Parallel, delayed
from torch_geometric.nn import GCNConv as PyG_GCNConv, VGAE as PyG_VGAE
from torch_geometric.data import Data
from scipy.sparse import lil_matrix
import pickle

/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from spektral.datasets import Cora

# Create a custom Dataset for the graph
class CoraDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        data = Cora()  # Load the dataset
        graph = data.graphs[0]  # Access the first graph in the dataset
        return [Graph(x=graph.x, a=graph.a, y=graph.y)]

In [4]:
from torch_geometric.datasets import Planetoid

# Create a custom Dataset for the graph
class CiteSeerDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = Planetoid(root=".", name="CiteSeer")  # Load CiteSeer dataset
        data = dataset[0]  # Access the first graph
        
        # Convert Torch tensors to NumPy
        x = data.x.numpy()
        edge_index = data.edge_index.numpy()
        y = data.y.numpy()

        # One-hot encode labels
        num_classes = y.max() + 1  # Number of classes
        y_one_hot = np.eye(num_classes)[y]  # One-hot encoding

        # Convert edge_index to a sparse adjacency matrix
        num_nodes = x.shape[0]
        adj = csr_matrix((num_nodes, num_nodes))  # Initialize sparse matrix
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[:, i]
            adj[src, dst] = 1
            adj[dst, src] = 1  # Ensure undirected graph

        return [Graph(x=x, a=adj, y=y_one_hot)]

In [5]:

# Create a custom Dataset for the graph
class PubMedDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = Planetoid(root=".", name="PubMed")  # Load PubMed dataset
        data = dataset[0]  # Access the first graph
        
        # Convert Torch tensors to NumPy
        x = data.x.numpy()
        edge_index = data.edge_index.numpy()
        y = data.y.numpy()

        # One-hot encode labels
        num_classes = y.max() + 1  # Number of classes
        y_one_hot = np.eye(num_classes)[y]  # One-hot encoding

        # Convert edge_index to a sparse adjacency matrix
        num_nodes = x.shape[0]
        adj = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[:, i]
            adj[src, dst] = 1
            adj[dst, src] = 1  # Ensure undirected graph

        return [Graph(x=x, a=adj, y=y_one_hot)]

In [6]:
from torch_geometric.datasets import Amazon

# Create a custom Dataset for DBLP
class AmazonPhotosDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = Amazon(".", name="photo")  # Load Amazon Computers dataset
        data = dataset[0]

        x = data.x.numpy()
        edge_index = data.edge_index.numpy()
        y = data.y.numpy()

        # One-hot encode labels
        num_classes = y.max() + 1
        y_one_hot = np.eye(num_classes)[y]

        # Convert edge_index to adjacency matrix
        num_nodes = x.shape[0]
        adj = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[:, i]
            adj[src, dst] = 1
            adj[dst, src] = 1  

        return [Graph(x=x, a=adj, y=y_one_hot)]


In [7]:
from torch_geometric.datasets import WikiCS

class WikiCSDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = WikiCS(root="./data/WikiCS")  # Download & load WikiCS dataset
        data = dataset[0]  # Access the graph

        # Features and labels
        x = data.x.numpy()
        y = data.y.numpy().flatten()

        # One-hot encode labels
        num_classes = y.max() + 1
        y_one_hot = np.eye(num_classes)[y]

        # Build adjacency matrix
        edge_index = data.edge_index.numpy()
        num_nodes = x.shape[0]
        adj = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[:, i]
            adj[src, dst] = 1
            adj[dst, src] = 1  # Ensure undirected

        return [Graph(x=x, a=adj, y=y_one_hot)]


In [8]:
# patch torch.load
torch_load_old = torch.load
def torch_load_patched(*args, **kwargs):
    kwargs["weights_only"] = False
    return torch_load_old(*args, **kwargs)
torch.load = torch_load_patched
from ogb.nodeproppred import NodePropPredDataset

class ArxivDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = NodePropPredDataset(name="ogbn-arxiv")
        graph, labels = dataset[0]

        x = graph["node_feat"]  # (num_nodes, num_features)
        a = sp.coo_matrix(
            (np.ones(graph["edge_index"].shape[1]), 
             (graph["edge_index"][0], graph["edge_index"][1])),
            shape=(x.shape[0], x.shape[0]),
        )

        # labels is shape (num_nodes, 1) with integer class indices
        labels = labels.squeeze()

        # Convert to one-hot encoded labels
        num_classes = labels.max() + 1
        y = np.eye(num_classes)[labels]   # (num_nodes, num_classes)

        return [Graph(x=x, a=a, y=y)]